# Generate AI Research Proposals - Baseline Condition

This notebook implements the baseline condition for AI-generated research proposals as outlined in the analysis plan.

## Steps:
1. Generate 23 research ideas from each AI model (GPT, Gemini, Claude) using the `generate_ideas_baseline` prompt
2. Save all titles and abstracts to a CSV file in `data/ai-proposals/baseline`
3. For each idea, generate a full proposal using the `generate_proposals` prompt with the same model
4. Add full proposals to the CSV file

## Setup and Install Dependencies

First, ensure all required packages are installed from `src/requirements.txt`.

In [1]:
# Option 1: Run setup.py (recommended - also creates .env config file)
# This uses the setup script from src/setup.py which installs from src/requirements.txt
import subprocess
import sys
import os

# Uncomment ONE of the following options:

# Option A: Run the full setup script (installs dependencies + creates .env file)
# subprocess.check_call([sys.executable, "src/setup.py"])

# Option B: Just install requirements from src/requirements.txt
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "src/requirements.txt"])

print("ℹ️  To install dependencies, uncomment one of the options above and run this cell")
print("✓ Dependencies defined in: src/requirements.txt")
print("✓ Setup script available at: src/setup.py")

ℹ️  To install dependencies, uncomment one of the options above and run this cell
✓ Dependencies defined in: src/requirements.txt
✓ Setup script available at: src/setup.py



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Import Required Modules

In [1]:
import sys
import os
import json
import pandas as pd
from pathlib import Path
from datetime import datetime
import logging

# Add src to path to import custom modules
src_path = os.path.join(os.getcwd(), 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Import custom modules from src/
from ai_models_interface import AIModelsInterface
from prompt_templates import PromptManager

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✓ Imports successful")
print(f"✓ Working directory: {os.getcwd()}")
print(f"✓ Python path includes: {src_path}")

✓ Imports successful
✓ Working directory: /Users/eveyhuang/Documents/NICO/human-AI-proposal-comparison
✓ Python path includes: /Users/eveyhuang/Documents/NICO/human-AI-proposal-comparison/src


## Load Configuration and Data

In [2]:
# Load call and information about NCEMS
with open('data/call_and_info.json', 'r') as f:
    call_and_info = json.load(f)

research_call = call_and_info['call']
ncems_info = call_and_info['info']

print("✓ Loaded call and NCEMS information")
print(f"\nResearch call preview: {research_call[:200]}...")
print(f"\nNCEMS info preview: {ncems_info[:200]}...")

✓ Loaded call and NCEMS information

Research call preview: This funding organization is dedicated to catalyzing multidisciplinary scientific teams to synthesize publicly available data to address fundamental questions related to emergence phenomena in the mol...

NCEMS info preview: The U.S. National Science Foundation National Synthesis Center for Emergence in the Molecular and Cellular Sciences (NCEMS) is catalyzing multidisciplinary scientific teams to leverage AI and data sci...


## Initialize AI Models Interface

In [3]:
# Initialize AI interface (will load API keys from config.env or .env file)
# The setup.py script creates .env in the root directory
ai_interface = AIModelsInterface(config_path='.env')

# Get available models
available_models = ai_interface.get_available_models()
print(f"✓ Available models: {available_models}")

# Define the models we want to use (GPT, Gemini, Claude)
models_to_use = ['gpt-5.2', 'gemini-3-pro-preview', 'claude-opus-4-5']
models_to_use = [m for m in models_to_use if m in available_models]
print(f"\nUsing models: {models_to_use}")

INFO:ai_models_interface:OpenAI GPT-4 initialized
INFO:ai_models_interface:Google Gemini initialized
INFO:ai_models_interface:Anthropic Claude initialized


✓ Available models: ['gpt-5.2', 'gemini-3-pro-preview', 'claude-opus-4-5']

Using models: ['gpt-5.2', 'gemini-3-pro-preview', 'claude-opus-4-5']


## Step 1: Generate Research Ideas (Titles and Abstracts)

Generate 23 research ideas from each AI model using the `generate_ideas_baseline` prompt.

In [4]:
# Initialize prompt manager
prompt_manager = PromptManager()

# Format the baseline prompt
baseline_prompt_template = prompt_manager.get_template('generate_ideas_baseline')

# Note: The template has {research_call}, {information_about_ncems}, and {num} placeholders
# We need to format it with our data
# Generate 23 ideas (same as total number of human proposals)
baseline_prompt = baseline_prompt_template.template.format(
    research_call=research_call,
    information_about_ncems=ncems_info,
    num=23
)

print("✓ Prompt template prepared")
print(f"✓ Configured to generate 23 research ideas per model")
print(f"\nPrompt preview (first 500 chars):\n{baseline_prompt}...")

✓ Prompt template prepared
✓ Configured to generate 23 research ideas per model

Prompt preview (first 500 chars):

            You are generating research ideas for a competitive funding call from NCEMS (National Synthesis Center for Emergence in Molecular and Cellular Sciences).

            RESEARCH CALL:
            This funding organization is dedicated to catalyzing multidisciplinary scientific teams to synthesize publicly available data to address fundamental questions related to emergence phenomena in the molecular and cellular biosciences. It supports community-scale synthesis and integration of existing molecular and cellular data to: Answer novel questions Solve long-standing puzzles in the molecular and cellular sciences Develop innovative research and analytical strategies Tap diverse new talent Train the future data-savvy workforce A community-scale synthesis project has three defining characteristics. First, it uses only existing publicly available data and often in

In [6]:
# Generate ideas from each model
all_ideas = []

for model_name in models_to_use:
    logger.info(f"\n{'='*60}")
    logger.info(f"Generating 23 research ideas using {model_name}...")
    logger.info(f"{'='*60}")
    
    try:
        # Generate content using the model
        response = ai_interface.generate_content(
            prompt=baseline_prompt,
            model_name=model_name,
            temperature=0.7,
            max_completion_tokens=16000
        )
        
        # Parse the JSON response
        try:
            # Find JSON in response
            response_text = response.strip()
            start_idx = response_text.find('{')
            end_idx = response_text.rfind('}') + 1
            
            if start_idx != -1 and end_idx > start_idx:
                json_str = response_text[start_idx:end_idx]
                ideas_data = json.loads(json_str)
                
                # Extract research ideas
                if 'research_ideas' in ideas_data:
                    research_ideas = ideas_data['research_ideas']
                    logger.info(f"✓ Generated {len(research_ideas)} ideas from {model_name}")
                    
                    # Add model name to each idea
                    for idea in research_ideas:
                        idea['model'] = model_name
                        idea['generated_at'] = datetime.now().isoformat()
                        all_ideas.append(idea)
                else:
                    logger.error(f"No 'research_ideas' key found in response from {model_name}")
            else:
                logger.error(f"Could not find valid JSON in response from {model_name}")
                
        except json.JSONDecodeError as e:
            logger.error(f"Failed to parse JSON from {model_name}: {e}")
            logger.error(f"Response preview: {response[:500]}...")
            
    except Exception as e:
        logger.error(f"Error generating ideas with {model_name}: {e}")

print(f"\n{'='*60}")
print(f"✓ Total ideas generated: {len(all_ideas)}")
print(f"{'='*60}")

INFO:__main__:
INFO:__main__:Generating 23 research ideas using gpt-5.2...
INFO:__main__:============================================================
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:✓ Generated 27 ideas from gpt-5.2
INFO:__main__:
INFO:__main__:Generating 23 research ideas using gemini-3-pro-preview...
INFO:__main__:============================================================
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:__main__:✓ Generated 23 ideas from gemini-3-pro-preview
INFO:__main__:
INFO:__main__:Generating 23 research ideas using claude-opus-4-5...
INFO:__main__:============================================================
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:__main__:✓ Generated 23 ideas from claude-opus-4-5



✓ Total ideas generated: 73


In [7]:
# Create DataFrame from ideas
ideas_df = pd.DataFrame(all_ideas)

# Reorder columns
column_order = ['model', 'title', 'abstract', 'generated_at']
ideas_df = ideas_df[column_order]

print(f"\nDataFrame shape: {ideas_df.shape}")
print(f"\nColumns: {list(ideas_df.columns)}")
print(f"\nIdeas per model:")
print(ideas_df['model'].value_counts())
print(f"\nFirst few rows:")
ideas_df.head()


DataFrame shape: (73, 4)

Columns: ['model', 'title', 'abstract', 'generated_at']

Ideas per model:
model
gpt-5.2                 27
gemini-3-pro-preview    23
claude-opus-4-5         23
Name: count, dtype: int64

First few rows:


,model,title,abstract,generated_at
0,gpt-5.2,A Cross-Cell-Type Atlas of Biomolecular Conden...,Biomolecular condensates (10–500 nm substructu...,2026-02-09T20:49:08.056375
1,gpt-5.2,Emergent Membrane Curvature Codes: Synthesizin...,Membrane contact sites (MCS) between organelle...,2026-02-09T20:49:08.056410
2,gpt-5.2,Mesoscale Cytoskeletal Percolation: A Synthesi...,Intracellular transport emerges from the colle...,2026-02-09T20:49:08.056412
3,gpt-5.2,Emergent Chromatin Nanodomain Topology: Integr...,Chromatin forms mesoscale nanodomains (50–300 ...,2026-02-09T20:49:08.056414
4,gpt-5.2,Protein Crowding as an Emergent Mesoscale Regu...,Cells exhibit emergent mesoscale physical prop...,2026-02-09T20:49:08.056415


In [8]:
# Create output directory if it doesn't exist
output_dir = Path('data/ai-proposals/baseline')
output_dir.mkdir(parents=True, exist_ok=True)

# Save ideas to CSV
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
ideas_file = output_dir / f'ai_ideas_baseline_{timestamp}.csv'
ideas_df.to_csv(ideas_file, index=False)

print(f"✓ Saved {len(ideas_df)} ideas to: {ideas_file}")

✓ Saved 73 ideas to: data/ai-proposals/baseline/ai_ideas_baseline_20260209_205423.csv


## Step 2: Generate Full Proposals from Ideas

For each idea, use the same AI model to generate a comprehensive research proposal using the `generate_proposals` prompt.

In [9]:
# Get the generate_proposals template
proposals_template = prompt_manager.get_template('generate_proposals')

print("✓ Loaded generate_proposals template")
print(f"\nTemplate parameters: {proposals_template.parameters}")

✓ Loaded generate_proposals template

Template parameters: ['title', 'abstract', 'research_call', 'information_about_ncems']


In [10]:
# Add columns for full proposal sections
ideas_df['background_and_significance'] = None
ideas_df['research_questions_and_hypotheses'] = None
ideas_df['methods_and_approach'] = None
ideas_df['expected_outcomes_and_impact'] = None
ideas_df['budget_and_resources'] = None
ideas_df['proposal_generated_at'] = None

print(f"✓ Added proposal columns to DataFrame")
print(f"New columns: {list(ideas_df.columns)}")

✓ Added proposal columns to DataFrame
New columns: ['model', 'title', 'abstract', 'generated_at', 'background_and_significance', 'research_questions_and_hypotheses', 'methods_and_approach', 'expected_outcomes_and_impact', 'budget_and_resources', 'proposal_generated_at']


In [11]:
# Generate full proposals for each idea
for idx, row in ideas_df.iterrows():
    model_name = row['model']
    title = row['title']
    abstract = row['abstract']
    
    logger.info(f"\n{'='*60}")
    logger.info(f"Generating proposal {idx+1}/{len(ideas_df)} using {model_name}")
    logger.info(f"Title: {title[:100]}...")
    logger.info(f"{'='*60}")
    
    try:
        # Format the proposal generation prompt
        proposal_prompt = proposals_template.template.format(
            research_call=research_call,
            information_about_ncems=ncems_info,
            title=title,
            abstract=abstract
        )
        
        # Generate proposal using the same model
        response = ai_interface.generate_content(
            prompt=proposal_prompt,
            model_name=model_name,
            temperature=0.7,
            max_completion_tokens=16000
        )
        
        # Parse the JSON response
        try:
            response_text = response.strip()
            start_idx = response_text.find('{')
            end_idx = response_text.rfind('}') + 1
            
            if start_idx != -1 and end_idx > start_idx:
                json_str = response_text[start_idx:end_idx]
                proposal_data = json.loads(json_str)
                
                # Extract proposal sections
                if 'proposal' in proposal_data:
                    proposal = proposal_data['proposal']
                    
                    # Update DataFrame with proposal sections
                    ideas_df.at[idx, 'background_and_significance'] = proposal.get('background_and_significance', '')
                    ideas_df.at[idx, 'research_questions_and_hypotheses'] = proposal.get('research_questions_and_hypotheses', '')
                    ideas_df.at[idx, 'methods_and_approach'] = proposal.get('methods_and_approach', '')
                    ideas_df.at[idx, 'expected_outcomes_and_impact'] = proposal.get('expected_outcomes_and_impact', '')
                    ideas_df.at[idx, 'budget_and_resources'] = proposal.get('budget_and_resources', '')
                    ideas_df.at[idx, 'proposal_generated_at'] = datetime.now().isoformat()
                    
                    logger.info(f"✓ Successfully generated proposal for: {title[:50]}...")
                else:
                    logger.error(f"No 'proposal' key in response for: {title[:50]}...")
            else:
                logger.error(f"Could not find valid JSON in response for: {title[:50]}...")
                
        except json.JSONDecodeError as e:
            logger.error(f"Failed to parse JSON for '{title[:50]}...': {e}")
            logger.error(f"Response preview: {response[:500]}...")
            
    except Exception as e:
        logger.error(f"Error generating proposal for '{title[:50]}...': {e}")
    
    # Save progress after each proposal (in case of interruption)
    if (idx + 1) % 5 == 0:  # Save every 5 proposals
        progress_file = output_dir / f'ai_proposals_baseline_progress_{timestamp}.csv'
        ideas_df.to_csv(progress_file, index=False)
        logger.info(f"✓ Progress saved to: {progress_file}")

print(f"\n{'='*60}")
print(f"✓ Completed generating all proposals")
print(f"{'='*60}")

INFO:__main__:
INFO:__main__:Generating proposal 1/73 using gpt-5.2
INFO:__main__:Title: A Cross-Cell-Type Atlas of Biomolecular Condensate Material States and Their Emergent Control of Gen...
INFO:__main__:============================================================
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:✓ Successfully generated proposal for: A Cross-Cell-Type Atlas of Biomolecular Condensate...
INFO:__main__:
INFO:__main__:Generating proposal 2/73 using gpt-5.2
INFO:__main__:Title: Emergent Membrane Curvature Codes: Synthesizing Cryo-ET, Proteomics, and Lipidomics to Predict Organ...
INFO:__main__:============================================================
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:✓ Successfully generated proposal for: Emergent Membrane Curvature Codes: Synthesizing Cr...
INFO:__main__:
INFO:__main__:Generating proposal 3/73 using gpt-5.2



✓ Completed generating all proposals


In [12]:
# Save final results
final_file = output_dir / f'ai_proposals_baseline_complete_{timestamp}.csv'
ideas_df.to_csv(final_file, index=False)

print(f"✓ Saved complete proposals to: {final_file}")
print(f"\nFinal DataFrame shape: {ideas_df.shape}")
print(f"\nProposals with all sections completed:")
print(ideas_df['background_and_significance'].notna().sum())

✓ Saved complete proposals to: data/ai-proposals/baseline/ai_proposals_baseline_complete_20260209_205423.csv

Final DataFrame shape: (73, 10)

Proposals with all sections completed:
53


## Summary Statistics

In [13]:
# Display summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

print(f"\nTotal ideas generated: {len(ideas_df)}")
print(f"\nIdeas per model:")
print(ideas_df['model'].value_counts())

print(f"\nProposals with completed sections:")
for col in ['background_and_significance', 'research_questions_and_hypotheses', 
            'methods_and_approach', 'expected_outcomes_and_impact', 'budget_and_resources']:
    completed = ideas_df[col].notna().sum()
    print(f"  {col}: {completed}/{len(ideas_df)} ({completed/len(ideas_df)*100:.1f}%)")

print(f"\nAverage abstract length: {ideas_df['abstract'].str.len().mean():.0f} characters")
print(f"\nOutput files saved in: {output_dir}")
print("\n" + "="*60)


SUMMARY STATISTICS

Total ideas generated: 73

Ideas per model:
model
gpt-5.2                 27
gemini-3-pro-preview    23
claude-opus-4-5         23
Name: count, dtype: int64

Proposals with completed sections:
  background_and_significance: 53/73 (72.6%)
  research_questions_and_hypotheses: 53/73 (72.6%)
  methods_and_approach: 53/73 (72.6%)
  expected_outcomes_and_impact: 53/73 (72.6%)
  budget_and_resources: 53/73 (72.6%)

Average abstract length: 2008 characters

Output files saved in: data/ai-proposals/baseline



In [14]:
# Display a sample proposal
print("\n" + "="*60)
print("SAMPLE PROPOSAL")
print("="*60)

sample_idx = 0
sample = ideas_df.iloc[sample_idx]

print(f"\nModel: {sample['model']}")
print(f"\nTitle: {sample['title']}")
print(f"\nAbstract: {sample['abstract'][:300]}...")
print(f"\nBackground (first 300 chars): {str(sample['background_and_significance'])[:300]}...")
print("\n" + "="*60)


SAMPLE PROPOSAL

Model: gpt-5.2

Title: A Cross-Cell-Type Atlas of Biomolecular Condensate Material States and Their Emergent Control of Gene Expression Noise

Abstract: Biomolecular condensates (10–500 nm substructures within micron-scale droplets) exhibit emergent material properties—viscosity, elasticity, percolation, and multiphase organization—that cannot be predicted from individual protein/RNA components. This project will test the hypothesis that condensate ...

Background (first 300 chars): Biomolecular condensates organize cellular biochemistry through mesoscale assemblies that sit between molecular interactions and organelle-scale structures. At the mesoscale (roughly 10–1000 nm), condensates exhibit emergent behaviors—viscoelasticity, percolation, arrested coarsening, and multiphase...



---
## Fix: Regenerate Missing Sections for Claude Proposals

16 out of 23 Claude proposals in `ai_proposals_baseline_complete_20260209_205423.csv` have empty
`background_and_significance`, `research_questions_and_hypotheses`, `methods_and_approach`,
`expected_outcomes_and_impact`, and `budget_and_resources` sections.

This section:
1. Identifies Claude rows with all sections missing
2. Regenerates those sections using `claude-opus-4-5` and the `generate_proposals` template
3. Replaces the rows in-place and overwrites the CSV

In [1]:
import sys, os, json, pandas as pd
from pathlib import Path
from datetime import datetime

src_path = os.path.join(os.getcwd(), 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from ai_models_interface import AIModelsInterface
from prompt_templates import PromptManager
from dotenv import load_dotenv
load_dotenv('.env', override=True)

# Load data
BASELINE_CSV = Path('data/ai-proposals/baseline/ai_proposals_baseline_complete_20260209_205423.csv')
df = pd.read_csv(BASELINE_CSV)

with open('data/call_and_info.json') as f:
    call_and_info = json.load(f)
research_call = call_and_info['call']
ncems_info = call_and_info['info']

# Identify Claude rows with all sections missing
SECTION_FIELDS = [
    'background_and_significance',
    'research_questions_and_hypotheses',
    'methods_and_approach',
    'expected_outcomes_and_impact',
    'budget_and_resources',
]

def all_sections_missing(row):
    return all(str(row[f]).strip() in ('', 'nan', 'None') for f in SECTION_FIELDS)

claude_mask = (df['model'] == 'claude-opus-4-5') & df.apply(all_sections_missing, axis=1)
to_fix = df[claude_mask].copy()
print(f'Found {len(to_fix)} Claude proposals with missing sections:')
for _, r in to_fix.iterrows():
    print(f'  [{r.name}] {r["title"][:80]}')

Found 16 Claude proposals with missing sections:
  [53] Metabolon Assembly and Emergent Metabolic Channeling: Integrating Structural, Me
  [54] Emergent Selectivity of Autophagy Receptors: Synthesizing Structural and Proteom
  [55] Emergent Timing in Circadian Protein Networks: Integrating Structural Dynamics w
  [56] Emergent Proton Gradients in Mitochondrial Cristae: Synthesizing Cryo-ET and Bio
  [57] Emergent Antigen Processing in the Immunoproteasome: Integrating Structural and 
  [58] Emergent Selectivity in Nuclear Export: Synthesizing Structural and Transport Da
  [59] Emergent Cooperativity in Transcription Factor Clusters: Synthesizing ChIP-seq, 
  [60] Emergent Viral Assembly: Synthesizing Cryo-EM and Sequence Data to Model How Cap
  [61] Emergent Membrane Curvature Sensing: Integrating Structural and Lipidomic Data t
  [62] Emergent Prion Strain Properties: Synthesizing Cryo-EM Structures with Clinical 
  [63] Emergent Cell Polarity from Protein Localization Networks: Synth

In [2]:
# Initialize Claude model and prompt template
ai_interface = AIModelsInterface(config_path='.env', override_env=True)
prompt_manager = PromptManager()
proposals_template = prompt_manager.get_template('generate_proposals')

MODEL = 'claude-opus-4-5'
assert MODEL in ai_interface.get_available_models(), f'{MODEL} not available — check ANTHROPIC_API_KEY in .env'
print(f'Using model: {MODEL}')
print(f'Template parameters: {proposals_template.parameters}')

INFO:ai_models_interface:OpenAI GPT-4 initialized
INFO:ai_models_interface:Google Gemini initialized
INFO:ai_models_interface:Anthropic Claude initialized


Using model: claude-opus-4-5
Template parameters: ['title', 'abstract', 'research_call', 'information_about_ncems']


In [3]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

def parse_proposal_json(response_text):
    """Extract proposal sections from JSON response. Returns dict or None."""
    s = response_text.strip()
    start = s.find('{')
    end = s.rfind('}') + 1
    if start == -1 or end <= start:
        return None
    try:
        data = json.loads(s[start:end])
        return data.get('proposal', data)  # handle both {proposal:{...}} and {...}
    except json.JSONDecodeError:
        return None

# Regenerate missing sections
n_fixed = 0
n_failed = 0

for idx, row in to_fix.iterrows():
    title = row['title']
    abstract = row['abstract']
    logger.info(f'[{idx}] Regenerating: {title[:70]}...')

    try:
        prompt = proposals_template.template.format(
            research_call=research_call,
            information_about_ncems=ncems_info,
            title=title,
            abstract=abstract,
        )
        response = ai_interface.generate_content(
            prompt=prompt,
            model_name=MODEL,
            temperature=0,
            max_tokens=16000,
        )
        proposal = parse_proposal_json(response)

        if proposal and any(proposal.get(f, '').strip() for f in SECTION_FIELDS):
            for field in SECTION_FIELDS:
                df.at[idx, field] = proposal.get(field, '')
            df.at[idx, 'proposal_generated_at'] = datetime.now().isoformat()
            n_fixed += 1
            logger.info(f'  ✓ Sections regenerated ({sum(len(str(proposal.get(f,"")).split()) for f in SECTION_FIELDS)} words total)')
        else:
            n_failed += 1
            logger.warning(f'  ✗ Could not parse sections from response. Preview: {response[:200]}')

    except Exception as e:
        n_failed += 1
        logger.error(f'  ✗ API error: {e}')

    # Checkpoint every 4 proposals
    if (n_fixed + n_failed) % 4 == 0:
        df.to_csv(BASELINE_CSV, index=False)
        logger.info(f'  Checkpoint saved ({n_fixed} fixed, {n_failed} failed so far)')

print(f'\n✓ Done: {n_fixed} fixed, {n_failed} failed')

INFO:__main__:[53] Regenerating: Metabolon Assembly and Emergent Metabolic Channeling: Integrating Stru...
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:__main__:  ✓ Sections regenerated (4388 words total)
INFO:__main__:[54] Regenerating: Emergent Selectivity of Autophagy Receptors: Synthesizing Structural a...
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:__main__:  ✓ Sections regenerated (3575 words total)
INFO:__main__:[55] Regenerating: Emergent Timing in Circadian Protein Networks: Integrating Structural ...
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:__main__:  ✓ Sections regenerated (3840 words total)
INFO:__main__:[56] Regenerating: Emergent Proton Gradients in Mitochondrial Cristae: Synthesizing Cryo-...
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:__main__:  ✓ Sections regenerated (3857 words tot


✓ Done: 16 fixed, 0 failed


In [4]:
# Save final result back to the same CSV (in-place replacement)
df.to_csv(BASELINE_CSV, index=False)
print(f'✓ Saved updated CSV to {BASELINE_CSV}')

# Verify: check remaining missing sections
df_check = pd.read_csv(BASELINE_CSV)
still_missing = (df_check['model'] == 'claude-opus-4-5') & df_check.apply(all_sections_missing, axis=1)
print(f'\nClaude proposals still missing all sections: {still_missing.sum()}')

# Word count summary for Claude after fix
claude_fixed = df_check[df_check['model'] == 'claude-opus-4-5']
for f in SECTION_FIELDS:
    wcs = claude_fixed[f].apply(lambda x: len(str(x).split()) if str(x).strip() not in ('','nan','None') else 0)
    print(f'  {f}: mean={wcs.mean():.0f} words, {(wcs==0).sum()} empty')

✓ Saved updated CSV to data/ai-proposals/baseline/ai_proposals_baseline_complete_20260209_205423.csv

Claude proposals still missing all sections: 0
  background_and_significance: mean=615 words, 0 empty
  research_questions_and_hypotheses: mean=780 words, 0 empty
  methods_and_approach: mean=963 words, 0 empty
  expected_outcomes_and_impact: mean=851 words, 0 empty
  budget_and_resources: mean=621 words, 0 empty
